# Networks

In [ ]:
import networkx as nx
import pandas as pd

In [ ]:
df = pd.read_json("../data/labelled/whole_pdf_docs.jsonl", lines=True)
# df = pd.read_json("../data/labelled/all.jsonl", lines=True) # TODO: merge B/I/O tags
df.head(5)

In [ ]:
from util import to_list_of_entities

# Convert list of ranges to list of tags
df["tags"] = [to_list_of_entities(df, i) for i in range(df.shape[0])]
df.head(5)

In [ ]:
from util import flatten

# Use unique sentences and entities
sentences = set(list(df["text"]))
entities = set(flatten(df["tags"]))

# Build list of links
links = []
for i, r in df.iterrows():
    row_links = []
    for tag in list(r["label"]):
        text = r["text"]
        start = tag[0]
        end = tag[1]
        label = tag[2]

        row_links.append((text, (text[start:end], label)))

    links += row_links

In [ ]:
print(f"Sentences: {len(sentences)}\nEntities: {len(entities)}\nLinks: {len(links)}")

In [ ]:
# Create a bipartite graph B: V_1 = sentences, V_2 = entities; V_1 -> V_2 iff v_2 appears in v_1
B = nx.Graph()

B.add_nodes_from(sentences, bipartite=0)
B.add_nodes_from(entities, bipartite=1)
B.add_edges_from(links)

In [ ]:
# Get entity nodes
top_nodes = {n for n, d in B.nodes(data=True) if d["bipartite"] == 1}
B_proj = nx.projected_graph(B, top_nodes)

In [ ]:
import matplotlib.pyplot as plt


ego = nx.ego_graph(
    B_proj,
    ("Enrico Fontana Paolo Conti", "PER"),
    radius=1,
)
plt.figure(figsize=(10, 10))
pos = nx.spring_layout(ego, seed=52)

# Get nodes based on NER tag
per_nodes = [n for n in ego.nodes if n[1] == "PER"]
loc_nodes = [n for n in ego.nodes if n[1] == "LOC"]
org_nodes = [n for n in ego.nodes if n[1] == "ORG"]
misc_nodes = [n for n in ego.nodes if n[1] == "MISC"]

nx.draw(
    nx.Graph(),
    pos,
    width=0.1,
    with_labels=True,
    font_size=5,
    node_size=100,
)

nx.draw_networkx_nodes(
    ego,
    pos,
    nodelist=per_nodes,
    node_size=100,
    node_color="y",
)

nx.draw_networkx_nodes(
    ego,
    pos,
    nodelist=org_nodes,
    node_size=100,
    node_color="g",
)

nx.draw_networkx_nodes(
    ego,
    pos,
    nodelist=loc_nodes,
    node_size=100,
    node_color="b",
)

nx.draw_networkx_nodes(
    ego,
    pos,
    nodelist=misc_nodes,
    node_size=100,
    node_color="black",
)

nx.draw_networkx_edges(
    ego,
    pos,
    ego.edges,
    width=0.1,
)

nx.draw_networkx_labels(ego, pos, font_size=5)

nx.draw_networkx_nodes(
    ego,
    pos,
    nodelist=[("Enrico Fontana Paolo Conti", "PER")],
    node_size=300,
    node_color="r",
)


In [ ]:
from operator import itemgetter

# Create a BA model graph - use seed for reproducibility
n = 1000
m = 2
seed = 20532
G = nx.barabasi_albert_graph(n, m, seed=seed)

# find node with largest degree
node_and_degree = G.degree()
(largest_hub, degree) = sorted(node_and_degree, key=itemgetter(1))[-1]

# Create ego graph of main hub
hub_ego = nx.ego_graph(G, largest_hub)

# Draw graph
pos = nx.spring_layout(hub_ego, seed=seed)  # Seed layout for reproducibility
nx.draw(hub_ego, pos, node_color="b", node_size=50, with_labels=False)

# Draw ego as large and red
# options = {"node_size": 300, "node_color": "r"}
# nx.draw_networkx_nodes(hub_ego, pos, nodelist=[largest_hub], **options)
# plt.show()